# 03 — Vision Redaction via AWS Bedrock
Send each page image to Claude claude-opus-4-6 on Bedrock.

PII/PHI is **replaced with realistic fictitious dummy values** (not blanked).
The same original value gets the **same replacement** across all pages of a document.

Outputs per page stored in `redacted_text/` as `{stem}_page_{n}.json`:
```json
{ "sanitized_text": "...", "mapping": [{"original_masked": ..., "replacement": ..., "type": ...}] }
```

In [ ]:
import subprocess, sys
for pkg in ["pymupdf", "boto3", "Pillow", "reportlab"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [ ]:
import boto3, json, base64, os, time
from pathlib import Path
from collections import defaultdict

from utils import get_logger, extract_json
logger = get_logger("03_redact_via_bedrock")

BASE_DIR      = Path(os.getcwd()).parent
TEMP_FOLDER   = BASE_DIR / "temp_images"
TEXT_FOLDER   = BASE_DIR / "redacted_text"

AWS_REGION    = "us-east-2"
BEDROCK_MODEL = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
MAX_TOKENS    = 4096
RETRY_LIMIT   = 3
RETRY_DELAY   = 5

TEXT_FOLDER.mkdir(exist_ok=True)
client = boto3.client("bedrock-runtime", region_name=AWS_REGION)
logger.info("Bedrock client ready — model: %s", BEDROCK_MODEL)

In [ ]:
def image_to_base64(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()


def redact_image(image_path: Path, prior_mapping: list[dict]) -> dict:
    """
    Send a page image to Bedrock.
    Returns dict: {sanitized_text: str, mapping: list[dict]}

    If the model returns non-JSON (e.g. for image-only/banner pages),
    the raw response is used as sanitized_text with an empty mapping
    rather than failing the entire pipeline.
    """
    logger.debug("Encoding image: %s", image_path.name)
    img_b64 = image_to_base64(image_path)
    prompt  = build_prompt(prior_mapping)

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": prompt}
            ]
        }]
    }

    last_exc = None
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            logger.debug("Invoking Bedrock (attempt %d/%d)", attempt, RETRY_LIMIT)
            resp = client.invoke_model(
                modelId=BEDROCK_MODEL, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            logger.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            return extract_json(raw)

        except json.JSONDecodeError as exc:
            # Model returned text but not valid JSON — retrying won't help.
            # Treat the raw text as sanitized content (no PII expected on image-only pages).
            logger.warning(
                "Page %s returned non-JSON response (likely image-only/banner page). "
                "Falling back to raw text with empty mapping. Preview: %s",
                image_path.name, exc.doc[:200].replace("\n", "\\n")
            )
            return {"sanitized_text": exc.doc.strip(), "mapping": []}

        except Exception as exc:
            last_exc = exc
            logger.warning("Attempt %d failed for %s: %s", attempt, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    logger.error("All %d attempts exhausted for %s", RETRY_LIMIT, image_path.name)
    raise last_exc


def merge_mapping(accumulated: list[dict], new_rows: list[dict]) -> list[dict]:
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        logger.debug("Mapping: +%d new entries (total %d)", added, len(accumulated))
    return accumulated


logger.info("Helper functions defined")

In [ ]:
def image_to_base64(image_path: Path) -> str:
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()


def redact_image(image_path: Path, prior_mapping: list[dict]) -> dict:
    """
    Send a page image to Bedrock.
    Returns dict: {sanitized_text: str, mapping: list[dict]}
    """
    logger.debug("Encoding image: %s", image_path.name)
    img_b64 = image_to_base64(image_path)
    prompt  = build_prompt(prior_mapping)

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": prompt}
            ]
        }]
    }

    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            logger.debug("Invoking Bedrock (attempt %d/%d)", attempt, RETRY_LIMIT)
            resp = client.invoke_model(
                modelId=BEDROCK_MODEL, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            logger.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            return extract_json(raw)
        except Exception as exc:
            logger.warning("Attempt %d failed for %s: %s", attempt, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)
            else:
                logger.error("All %d attempts exhausted for %s", RETRY_LIMIT, image_path.name)
                raise


def merge_mapping(accumulated: list[dict], new_rows: list[dict]) -> list[dict]:
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        logger.debug("Mapping: +%d new entries (total %d)", added, len(accumulated))
    return accumulated


logger.info("Helper functions defined")

In [ ]:
# Group images by PDF stem
image_files = sorted(TEMP_FOLDER.glob("*.png"))
logger.info("Found %d page image(s) in temp_images/", len(image_files))

grouped: dict[str, list[Path]] = defaultdict(list)
for img in image_files:
    grouped[img.stem.rsplit("_page_", 1)[0]].append(img)

for stem in grouped:
    grouped[stem].sort(key=lambda p: int(p.stem.rsplit("_page_", 1)[-1]))
    logger.info("  %s: %d page(s)", stem, len(grouped[stem]))

In [ ]:
# Run redaction — skips pages that already have a cached .json file
for pdf_stem, page_images in grouped.items():
    logger.info("=== Processing PDF: %s (%d pages) ===", pdf_stem, len(page_images))
    accumulated_mapping: list[dict] = []

    for img_path in page_images:
        cache = TEXT_FOLDER / f"{img_path.stem}.json"

        if cache.exists():
            logger.info("[cache] %s", img_path.name)
            cached = json.loads(cache.read_text(encoding="utf-8"))
            accumulated_mapping = merge_mapping(accumulated_mapping, cached.get("mapping", []))
            continue

        logger.info("Redacting %s ...", img_path.name)
        result = redact_image(img_path, prior_mapping=accumulated_mapping)
        cache.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        accumulated_mapping = merge_mapping(accumulated_mapping, result.get("mapping", []))
        logger.info("Done: %s — %d mapping entries this page", img_path.name, len(result.get("mapping", [])))

    logger.info("PDF complete: %s — %d unique replacements total", pdf_stem, len(accumulated_mapping))

logger.info("Redaction finished for all PDFs")

In [ ]:
# Preview: first page sanitized text + mapping table for first PDF
json_files = sorted(TEXT_FOLDER.glob("*.json"))
if not json_files:
    logger.warning("No output files found")
else:
    first = json.loads(json_files[0].read_text(encoding="utf-8"))
    logger.info("Preview: %s", json_files[0].name)
    print(f"\n=== {json_files[0].name} — sanitized text ===")
    print(first["sanitized_text"])

    first_stem = json_files[0].stem.rsplit("_page_", 1)[0]
    all_rows: list[dict] = []
    for jf in sorted(TEXT_FOLDER.glob(f"{first_stem}_page_*.json"),
                     key=lambda p: int(p.stem.rsplit("_page_", 1)[-1])):
        data = json.loads(jf.read_text(encoding="utf-8"))
        for row in data.get("mapping", []):
            key = (row["original_masked"], row["type"])
            if key not in {(r["original_masked"], r["type"]) for r in all_rows}:
                all_rows.append(row)

    print(f"\n=== Summary table — {first_stem} ({len(all_rows)} entries) ===")
    print(f"{'Original (masked)':<28} {'Replacement':<28} {'Type'}")
    print("-" * 70)
    for row in all_rows:
        print(f"{row['original_masked']:<28} {row['replacement']:<28} {row['type']}")